## **Entrenamiento de TransientNeRF**

Este cuaderno usa el flujo original del repositorio mediante `train.py` y `loader_synthetic.py`. El experimento predeterminado entrena con cinco vistas y evalúa las veinte restantes.

## Datos necesarios

La funcion "loader" lee directamente `images`, `poses` y `transients` desde `data/scene_0.h5`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
DATASET = ROOT / 'data' / 'scene_0.h5'
assert (ROOT / 'train.py').is_file(), 'Abra el cuaderno desde la raíz del repositorio.'
assert DATASET.is_file(), f'No se encontró el conjunto de datos: {DATASET}'
print('Python:', sys.executable)
print('Datos:', DATASET)
# !{sys.executable} -m pip install -r requirements.txt

In [ ]:
import h5py
import numpy as np

with h5py.File(DATASET, 'r') as f:
    print('Atributos:', dict(f.attrs))
    for name in ('images', 'poses', 'transients'):
        print(f'{name}: forma={f[name].shape}, tipo={f[name].dtype}')
    assert f['transients'].shape == (25, 256, 256, 1034, 3)
    assert f.attrs['pose_type'] == 'camera_to_world'

focal = 128 / np.tan(np.deg2rad(30))
K = np.array([[focal, 0, 128], [0, focal, 128], [0, 0, 1]])
print('Matriz intrínseca K:\n', K)

In [ ]:
NUM_VIEWS = 5  # Use 2, 3 o 5.
MAX_STEPS = 10000
DEVICE = 'cuda:0'
SCENE_AABB = '[-1.8, -0.05, -1.8, 1.8, 3.55, 1.8]'

train_ids = np.rint(np.linspace(0, 24, NUM_VIEWS)).astype(int)
test_ids = np.setdiff1d(np.arange(25), train_ids)
print('Vistas de entrenamiento:', train_ids.tolist())
print('Vistas reservadas:', len(test_ids))

In [ ]:
CONFIG = ROOT / 'configs' / 'train' / 'simulated' / f'scene_0_{NUM_VIEWS}views.ini'
CONFIG.write_text(f'''exp_name = "scene_0_{NUM_VIEWS}views"
version = "simulated"
data_root_fp = "./data/scene_0.h5"
num_views = {NUM_VIEWS}
n_bins = 1034
img_shape = 256
img_shape_test = 256
aabb = "{SCENE_AABB}"
exposure_time = 0.009
start_opl = 10.1
tfilter_sigma = 3
rfilter_sigma = 0.15
num_rays_per_batch = 512
render_n_samples = 4096
grid_resolution = 128
grid_nlvl = 1
near_plane = 0
far_plane = 20
alpha_thre = 0
occ_thre = 0.01
space_carving = 0.007
lr = 1e-3
max_steps = {MAX_STEPS}
steps_til_checkpoint = 50000
sample_as_per_distribution = "False"
exp = "True"
final = "True"
outpath = "./results"
pixels_to_plot = ["(128, 128)", "(96, 128)", "(128, 96)"]
img_scale = 100
seed = 42
device = "{DEVICE}"
''')
print(CONFIG)

In [ ]:
from loaders.loader_synthetic import SubjectLoaderTransient

dataset = SubjectLoader(subject_id='scene_0', root_fp=str(DATASET), split='train',
    num_rays=4, img_shape=(256, 256), n_bins=1034, num_views=NUM_VIEWS)
dataset.rep = 1
sample = dataset[0]
print('Vistas seleccionadas:', dataset.view_ids.tolist())
print('Distancia focal:', dataset.focal)
print('Píxeles:', sample['pixels'].shape, 'Rayos:', sample['rays'].origins.shape)
assert sample['pixels'].shape == (4, 1034, 3)
assert sample['pixels'].isfinite().all()

In [ ]:
import subprocess
Path('results').mkdir(exist_ok=True)
smoke_cmd = [sys.executable, 'train.py', '-c', str(CONFIG), '--max_steps', '100',
             '--steps_til_checkpoint', '100', '--final', 'True',
             '--exp_name', f'scene_0_{NUM_VIEWS}views_smoke']
print(' '.join(smoke_cmd))
# subprocess.run(smoke_cmd, check=True)

In [ ]:
train_cmd = [sys.executable, 'train.py', '-c', str(CONFIG)]
print(' '.join(train_cmd))
# subprocess.run(train_cmd, check=True)

In [ ]:
# subprocess.Popen(['tensorboard', '--logdir', 'results', '--port', '6006'])